In [ ]:
from config import *

# GRPO算法的流程
## vllm实现
+ 初始化old policy
+ 利用old policy采样Q个问题，然后进行rollout，每个问题对应一个组(G个)
+ 计算奖励函数/优势函数
## 迭代n步batch进行梯度更新
+ log_probs: 这个很容易, `get_response_log_probs`
+ old_log_probs: 需要定义新的vllm函数.`get_response_log_probs`

## 数据集
+ vllm进行rollout时，输入应该是List[List[str]]，因此数据集应该返回的是prompts, ground_truths, answers
+ 评估时：仍然使用SFTDataset，使用AutoModel进行评估。

## 配置文件

In [2]:
# 训练模型
model_name ='../model/Qwen2.5-Math-1.5B'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path=model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map=get_device(4),# device 1
)

GRPODataset
+ 预期返回rollout_batch_size大小的样本 rollout_batch_size = n_prompts * rollout_size(group-size)

In [2]:
# 返回vllm实例
model_name ='../model/Qwen2.5-Math-1.5B'
vllm = init_vllm(model_name,device=get_device(5),seed=42,gpu_memory_utilization=0.6)# device 1
sampling_params = SamplingParams(
    temperature=1.0,
    top_p=1.0,
    stop=['</answer>'],
    seed=42,
    include_stop_str_in_output=True,
    max_tokens=1024,
    min_tokens=32,
    n=8,
)

INFO 03-12 18:16:03 config.py:542] This model supports multiple tasks: {'classify', 'score', 'generate', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 03-12 18:16:03 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='../model/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='../model/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda:5, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=../model/Qwen2.5-Math-1.5B, nu

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-12 18:16:12 model_runner.py:1115] Loading model weights took 0.0000 GB
INFO 03-12 18:16:14 worker.py:267] Memory profiling takes 1.31 seconds
INFO 03-12 18:16:14 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.60) = 14.15GiB
INFO 03-12 18:16:14 worker.py:267] model weights take 0.00GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 0.00GiB; the rest of the memory reserved for KV Cache is 14.15GiB.
INFO 03-12 18:16:14 executor_base.py:110] # CUDA blocks: 33120, # CPU blocks: 9362
INFO 03-12 18:16:14 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 129.38x
INFO 03-12 18:16:24 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:53<00:00,  1.52s/it]

INFO 03-12 18:17:18 model_runner.py:1562] Graph capturing finished in 53 secs, took 0.00 GiB
INFO 03-12 18:17:18 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 65.83 seconds


In [ ]:
# GRPO实验配置
config = GRPOConfig.from_json('../cs336_alignment/configs/grpo_math.json')
config.prompt_template_path

✅ GRPOConfig loaded from ../cs336_alignment/configs/grpo_math.json


'cs336_alignment/prompts/r1_zero.prompt'

: 

In [13]:
rollout_prompts, rollout_responses, advantages = dataset.get_rollout_batch()

Processed prompts:  12%|█▎        | 64/512 [01:10<08:10,  1.09s/it, est. speed input: 156.45 toks/s, output: 2768.00 toks/s]


In [15]:
rollout_prompts[:8], rollout_responses[:8], advantages[:8]
# rollout_prompts[8:16], rollout_responses[8:16], advantages[8:16]

(['A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Expand the following expression: $(13x+15)\\cdot 2x$\nAssistant: <think>',
  'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Expand the following expression: $(13x+15)\\cdot 2x$\nAssistant: <think>',
  

生成GRPO数据
+ 返回一个字典,包括原始数据(b g ?)和展开后的数据 b*g ?

In [2]:
model_name ='../model/Qwen2.5-Math-1.5B'
vllm = init_vllm(model_name,device=get_device(5),seed=42,gpu_memory_utilization=0.6)
sampling_params = SamplingParams(
    temperature=1.0,
    top_p=1.0,
    stop=['</answer>'],
    seed=42,
    include_stop_str_in_output=True,
    max_tokens=256,
    min_tokens=32,
    n=2,
    logprobs=1,# 输出前4个的logprobs,
)

INFO 03-09 21:31:35 config.py:542] This model supports multiple tasks: {'embed', 'generate', 'score', 'classify', 'reward'}. Defaulting to 'generate'.
INFO 03-09 21:31:35 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='../model/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='../model/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda:5, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=../model/Qwen2.5-Math-1.5B, nu

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-09 21:31:37 model_runner.py:1115] Loading model weights took 0.0000 GB
INFO 03-09 21:31:38 worker.py:267] Memory profiling takes 0.70 seconds
INFO 03-09 21:31:38 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.60) = 14.15GiB
INFO 03-09 21:31:38 worker.py:267] model weights take 0.00GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 0.00GiB; the rest of the memory reserved for KV Cache is 14.15GiB.
INFO 03-09 21:31:38 executor_base.py:110] # CUDA blocks: 33120, # CPU blocks: 9362
INFO 03-09 21:31:38 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 129.38x
INFO 03-09 21:31:41 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:15<00:00,  2.22it/s]

INFO 03-09 21:31:57 model_runner.py:1562] Graph capturing finished in 16 secs, took 0.00 GiB
INFO 03-09 21:31:57 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 19.93 seconds


In [27]:
# group_size = 2
prompts = ['Who are u', 'What is your name']
unflatten_metdata, flatten_metadata = generate_grpo_samples(
    vllm,
    prompts,
    sampling_params,
)

Processed prompts:  50%|█████     | 2/4 [00:01<00:01,  1.13it/s, est. speed input: 3.97 toks/s, output: 377.47 toks/s]


In [35]:
flatten_responses = flatten_metadata['responses']
flatten_output_logprobs = flatten_metadata['output_logprobs']
repeated_prompt_token_ids = flatten_metadata['prompt_token_ids']
responses = unflatten_metdata['responses']
output_logprobs = unflatten_metdata['output_logprobs']
prompt_token_ids = unflatten_metdata['prompt_token_ids']
len(flatten_responses) ,len(flatten_output_logprobs), len(repeated_prompt_token_ids)
repeated_prompt_token_ids,flatten_responses[0],flatten_responses[1]

([[15191, 525, 575],
  [15191, 525, 575],
  [3838, 374, 697, 829],
  [3838, 374, 697, 829]],
 'ỡ Taieb, and what is their field of research?\n\nuỡ Taieb is a distinguished researcher in the field of computer science, with a particular focus on knowledge representation and automated reasoning. Their work has significantly contributed to the advancement of artificial intelligence, as evidenced by their publications in reputable journals such as "Theory and Practice of Logic Programming." Their research interests include the development of logical systems and the application of computational techniques to solve complex problems in AI.',
 'SeDin Users and what are their properties?\n\nuSeDin Users are members who serve upwards of 5 students each, which is defined as a quarter load. These users iSTC UBUs on MAC and PC-testing network are usingFcn和睦的 Windows. They play⌡ auf等一批_games to play1,2 sudokulous11 210, story of1,2 gluottas, flu솀好莱坞 20b48ab 123,, kills 40, clockball, 183, 2, 11 1 100

In [37]:
model_name ='../model/Qwen2.5-Math-1.5B'
tokenizer = AutoTokenizer.from_pretrained(model_name)
inputs = tokenize_prompt_and_output(prompts, responses, tokenizer)
input_ids = inputs['input_ids'].to(get_device(4))
labels = inputs['labels'].to(get_device(4))
response_mask = inputs['response_mask'].to(get_device(4))
response_mask.shape # b l

torch.Size([2, 351])

In [2]:
@torch.no_grad()
def get_response_log_probs_vllm(
        repeated_prompt_ids:List[List[int]],# b*g ?
        flatten_output_log_probs:List[List[float]],# b*g ?
        response_mask: torch.Tensor, # ... l
    )->torch.Tensor:
        """
        return : bg l
        """
        max_seq_len = response_mask.shape[-1]
        old_log_probs = [
                [-100.0]* len(prompt_ids) + old_log_probs + 
                [-100.0]* (max_seq_len -len(prompt_ids) - len(old_log_probs)+1) # padding last token
                for prompt_ids, old_log_probs in zip(repeated_prompt_ids, flatten_output_log_probs)
        ]
        # b*g l+1 多出一个token,补充一个padding
        old_log_probs = torch.tensor(old_log_probs, device=response_mask.device)
        return old_log_probs[:,1:]# b*g l

验证GRPOTrainer
+ init: self.training_set是不是GRPODataset类型

In [ ]:
# 训练模型
model_name ='../model/Qwen2.5-Math-1.5B'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path=model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map=get_device(4),# device 1
)
vllm = init_vllm(model_name,device=get_device(5),seed=42,gpu_memory_utilization=0.6)

INFO 03-09 22:56:11 config.py:542] This model supports multiple tasks: {'classify', 'score', 'embed', 'reward', 'generate'}. Defaulting to 'generate'.
INFO 03-09 22:56:11 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='../model/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='../model/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda:5, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=../model/Qwen2.5-Math-1.5B, nu

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-09 22:56:13 model_runner.py:1115] Loading model weights took 0.0000 GB
INFO 03-09 22:56:14 worker.py:267] Memory profiling takes 0.70 seconds
INFO 03-09 22:56:14 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.60) = 14.15GiB
INFO 03-09 22:56:14 worker.py:267] model weights take 0.00GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 0.00GiB; the rest of the memory reserved for KV Cache is 14.15GiB.
INFO 03-09 22:56:15 executor_base.py:110] # CUDA blocks: 33120, # CPU blocks: 9362
INFO 03-09 22:56:15 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 129.38x
INFO 03-09 22:56:18 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:16<00:00,  2.16it/s]

INFO 03-09 22:56:34 model_runner.py:1562] Graph capturing finished in 16 secs, took 0.00 GiB
INFO 03-09 22:56:34 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 20.92 seconds


: 

In [ ]:
trainer = GRPOTrainer(
    model = model
    tokenizer = tokenizer,
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5),
    
)